In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
import jax
from scipy.stats import gaussian_kde

np.random.seed(42)

#data_dir = "/project/astro/sbi_lab/data/"
data_dir = "."

true_AB = [1.0, 2.5]

In [ ]:
n_pix = 64
res = np.load(os.path.join(data_dir, f"data_{n_pix=}.npz"))

In [ ]:
fields, data, k, powers, modes, data_powerspec, data_modes, parameters = res["fields"], res["data"], res["k"], res["powers"], res["modes"], res["data_powerspec"], res["data_modes"], res["parameters"]

In [ ]:
# Some functions you might find useful
# 1) Determine the contour levels from given probability levels
def get_levels(z, levels=[0, 0.68, 0.95]):
    # Get flattened posterior grid
    z_flattened = sorted(z.flatten(), reverse=True)
    # Get the cumulative sum over the posterior (equals 1/(da db) for normalzied posterior)
    z_cum = np.cumsum(z_flattened)
    z_cum /= z_cum[-1]
    # Normalize to the posterior sum = 1, then get the levels where the cumulative is exceeding a given level
    thresholds = []
    for level in levels:
        # Those posterior values at which that happens are added to the level
        idx = np.argmin(np.abs(z_cum - level))
        thresholds.append(z_flattened[idx])
    # return not just the levels, but also 0 so that matplotlib has a lowest level as well
    return [0]+sorted(thresholds)
# 2) Plotting a grid of likelihood evaluations, when evaluated over A_range and B_range. ax = matplotlib.axes object you want to plot onto
def plot_loglikelihood_grid(ax, A_range, B_range, ln_L_grid, true_AB = [1.0, 2.5]):
    # step a) converting lnL to L
    L_max = np.max(ln_L_grid)
    unnorm_posterior = np.exp(ln_L_grid - L_max)
    # step b) normalizing by integral
    da = A_range[1] - A_range[0]
    db = B_range[1] - B_range[0]
    posterior = unnorm_posterior/np.sum(unnorm_posterior)/da/db
    # step c) generating a meshgrid for the plot, obtaining levels
    AA, BB = np.meshgrid(A_range, B_range, indexing='ij')
    lvls = get_levels(posterior)
    # step d) plotting given the levels
    #ax.contourf(AA, BB, posterior, levels=lvls)
    ax.contourf(AA, BB, posterior)
    # step e) optionally plotting the true vlaues
    if true_AB:
        ax.axvline(true_AB[0], color="red")
        ax.axhline(true_AB[1], color="red")

# 3) Plotting a Fisher matrix ellispe, with inverse (!) fisher matrix cov, central point center, and ax = matplotlib.axes object you want to plot onto
from matplotlib.patches import Ellipse
def plot_fisher_ellipse(ax, cov, center, color='black', label='68% CL Fisher'):
    # Get eigenvectors to know orientation, eigenvalues to know size
    vals, vecs = np.linalg.eigh(cov)

    # Order them in order to make sure that we plot biggest eigenvalue as longest direction
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    # Compute angle, width, height (2.3 * eigenvalue = ellipse size, since 2D chi^2 distribution at 2.3 reaches 68% CL)
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(2.30 * vals)

    # Finally draw the actual ellipse
    ellipse = Ellipse(xy=center, width=width, height=height, angle=theta,
                      edgecolor=color, fc='none', lw=2, label=label)
    ax.add_patch(ellipse)

### Example : The analytical likelihood plotted, in a single command

In [ ]:
A_range = np.linspace(0.1, 2.0, 100)  # 100 points for Amplitude
B_range = np.linspace(2.0, 3.0, 101)   # 101 points for Tilt
AA, BB = np.meshgrid(A_range, B_range, indexing='ij') # Two grids, once only A values, once only B values. Together, they form coordinates coordinates
# The positions are = np.vstack([AA.ravel(), BB.ravel()])
print(np.vstack([AA.ravel(), BB.ravel()]).T)

In [ ]:
# Fill now the variable
# ln_L_grid with your grid of ln L evaluations for the analytical likelihood # ln_L_grid[i,j] = log-likelihood(A[i], B[j])

In [ ]:
plot_loglikelihood_grid(plt.gca(), A_range, B_range, ln_L_grid)

### To get the MLE, we can solve some simple equations. Please check that you understand how you would do it generally
 In this case it happens to be that we can simplify the MLE equations and obtain A as a function of B analytically.
 For B we then need to actually solve the equations, leading to the below function to be set zero (estimating_equation)
 We also give a small gueess that is close enough from a polynomial fit

 Then we finally obtain A analytically as a function of B
 Finally we return both values

In [ ]:
from scipy.optimize import fsolve

def solve_mle(k, p_obs, n_modes):
    ln_k = np.log(k)
    target_mean_log = np.sum(n_modes * ln_k) / np.sum(n_modes)

    def estimating_equation(B):
        weights = n_modes * p_obs * np.power(k, B)
        weighted_mean_log = np.sum(weights * ln_k) / np.sum(weights)
        return weighted_mean_log - target_mean_log

    # Initial guess
    b_guess = -np.polyfit(ln_k, np.log(p_obs), 1)[0]
    b_mle = fsolve(estimating_equation, x0=b_guess)[0]
    a_mle = np.sum(n_modes * p_obs * np.power(k, b_mle)) / np.sum(n_modes)

    return a_mle, b_mle

In [ ]:
mle_data = np.array(solve_mle(k, data_powerspec, data_modes))
print(mle_data)
a_mle, b_mle = mle_data

In [ ]:
# Insert here your components of the Fisher matrix
f_aa = "Insert your code here"
f_bb = "Insert your code here"
f_ab = "Insert your code here"

fisher_matrix = np.array([[f_aa, f_ab],[f_ab, f_bb]])
cov_fisher = np.linalg.inv(fisher_matrix)

fig = plt.figure()
ax = plt.gca()
plot_loglikelihood_grid(ax, A_range, B_range, ln_L_grid)
plot_fisher_ellipse(ax, cov_fisher, np.array([a_mle, b_mle]))

In [ ]:
mles = np.array([solve_mle(k, p, m) for p, m in zip(powers, modes)])

In [ ]:
# Your code for plotting the differences between MLEs and parameters here!

In [ ]:
from library import SimpleMLP, SimpleCNN, CNNEnsemble, Trainer
import jax.random as jr
key = jr.PRNGKey(42)

dat = fields # Which training data to feed into the compression network
dat_data = data # Which mock data to try to measure/compress with the compression network

# If we use the full 64x64 images, these are two options, the latter working better. Try out your own if you want!
# model = SimpleMLP(in_size=n_pix*n_pix, out_size=2, width_size=64, depth=3, key=key)
model = CNNEnsemble(key, n_models=20)

# If we instead use the log power spectra, then we might be able to get away with a simpler network. Let's try out if you want!
# dat = np.log(powers)
# dat_data = np.log(data_powerspec)
# model = SimpleMLP(in_size=50, out_size=2, width_size=64, depth=3, key=key)


# Whatever you choose, it must be trained

trainer = Trainer(model, lr=1e-3)

compressor = trainer.fit(dat, parameters, steps=2_000)

In [ ]:
compressed_params = jax.vmap(compressor)(dat)

In [ ]:
compressed_data = compressor(dat_data)
print(compressed_data)

In [ ]:
def plot_diagonal(x):
    plt.plot([np.min(x), np.max(x)],[np.min(x), np.max(x)], color="black")

plt.scatter(parameters[:,0], compressed_params[:, 0])
plot_diagonal(parameters[:,0])
plt.figure()
plt.scatter(parameters[:,1], compressed_params[:, 1])
plot_diagonal(parameters[:,1])

In [ ]:
delta_params = parameters-compressed_params
# Your plot to show them here!

In [ ]:
# Write your CNF fitting here

In [ ]:
# Cool code to evaluate the CNF over all the input A,B values of a grid, if you happen to need it
#packed_likelihood = lambda A,B: trained_cnf.single_log_prob_x(compressed_data, jnp.array([A,B]), key)
#grid_fn = jax.vmap(
#    jax.vmap(packed_likelihood, in_axes=(None, 0)), 
#    in_axes=(0, None)
#)
#ln_L_grid_cnf = grid_fn(jnp.array(A_range), jnp.array(B_range))

In [ ]:
# Write your KDE fitting below

In [ ]:
concatenated_vectors = np.array([np.concatenate([x,y]) for (x,y) in zip(compressed_params, parameters)])

In [ ]:
kde = gaussian_kde(concatenated_vectors.T)

In [ ]:
# A method to evaluate the KDE over the whole A,B grid in case you find it useful
#ab_pairs = np.column_stack([AA.ravel(), BB.ravel()])
#summary_tiled = np.tile(compressed_data.flatten(), (ab_pairs.shape[0], 1))
#full_input_matrix = np.hstack([summary_tiled, ab_pairs])
#unnorm_kde = kde(full_input_matrix.T).reshape(AA.shape)